> 在线讲义：[第 47 章 · MAE 与 DINO：不要标签也能学会看图](https://wikieden.github.io/beyond-transformer/lessons/47-mae-dino.html) · 本书《超越 Transformer》

# 第 47 章 · MAE 与 DINO：不要标签也能学会看图

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/47-mae-dino.html) · 仓库里 `beyond-transformer/notes/lessons/47-mae-dino.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 遮挡 | 剩下 = 196 × (1 − r) | `random_mask(196, 0.75, seed)` | 留 49 块，遮 147 块；格子 2401，是 38416 的 1/16 |
| 第二步 打乱再还原 | shuffle / unshuffle | `shuffle_and_restore(...)` | 还原后顺序和原来完全一样 |
| 第三步 loss | 归一化 + MSE | `mae_loss([2, 4, 6, 8], [-1, -0.5, 0.5, 1])` | 目标 [−1.342, −0.447, 0.447, 1.342]，MSE ≈ 0.060 |
| 第四步 DINO | 式 3 | `dino_loss(...)` | $P_t$ = [0.468, 0.468, 0.063]，$P_s$ = [0.245, 0.665, 0.090]，H ≈ 1.003 |
| 第五步 EMA | $\lambda\theta_t + (1-\lambda)\theta_s$ | `ema(1.0, 2.0, 0.996)` | 1.004 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `random_mask`**：MAE 随机遮挡：打乱 n 个位置，前面的留给 encoder，后面的遮掉。

In [ ]:
def random_mask(n, ratio, seed=0):
    """MAE 随机遮挡：打乱 n 个位置，前面的留给 encoder，后面的遮掉。"""
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    n_mask = int(round(n * ratio))
    return np.sort(perm[: n - n_mask]), np.sort(perm[n - n_mask :])

**函数 `shuffle_and_restore`**：MAE 的实现：打乱 → 砍掉后面 → 接上 mask token → 按原顺序还原。返回还原后的序列（被遮处为 None）。

In [ ]:
def shuffle_and_restore(tokens, seed=0, ratio=0.75):
    """MAE 的实现：打乱 → 砍掉后面 → 接上 mask token → 按原顺序还原。返回还原后的序列（被遮处为 None）。"""
    rng = np.random.default_rng(seed)
    n = len(tokens)
    perm = rng.permutation(n)
    keep = perm[: n - int(round(n * ratio))]
    seq = [tokens[i] for i in keep] + [None] * (n - len(keep))  # encoder 输出 + mask token
    order = list(keep) + [i for i in perm if i not in set(keep)]
    restored = [None] * n
    for pos, tok in zip(order, seq):
        restored[pos] = tok
    return restored

**函数 `mae_loss`**：MAE loss：目标先按这一块自己的平均、标准差归一化，再算均方误差。

In [ ]:
def mae_loss(x, pred):
    """MAE loss：目标先按这一块自己的平均、标准差归一化，再算均方误差。"""
    x = np.asarray(x, dtype=float)
    target = (x - x.mean()) / x.std()
    return target, ((np.asarray(pred) - target) ** 2).mean()

**函数 `softmax`**：softmax（先减最大值）。

In [ ]:
def softmax(v):
    """softmax（先减最大值）。"""
    e = np.exp(v - v.max())
    return e / e.sum()

**函数 `dino_loss`**：DINO 式 3：老师输出减中心 c、除以 τt 做 softmax；学生除以 τs 做 softmax；交叉熵 H(Pt, Ps)。

In [ ]:
def dino_loss(t, s, c, tau_t, tau_s):
    """DINO 式 3：老师输出减中心 c、除以 τt 做 softmax；学生除以 τs 做 softmax；交叉熵 H(Pt, Ps)。"""
    p_t = softmax((np.asarray(t, float) - c) / tau_t)
    p_s = softmax(np.asarray(s, float) / tau_s)
    return p_t, p_s, -(p_t * np.log(p_s)).sum()

**函数 `ema`**：老师 = λ·老师 + (1 − λ)·学生。

In [ ]:
def ema(theta_t, theta_s, lam):
    """老师 = λ·老师 + (1 − λ)·学生。"""
    return lam * theta_t + (1 - lam) * theta_s

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 遮挡：196 块，遮 75%")
keep, drop = random_mask(196, 0.75)
print("留给 encoder：", len(keep), "块；遮掉：", len(drop), "块")
print("注意力格子：", len(keep) ** 2, "，不遮时", 196**2, "，比值 1 /", 196**2 // len(keep) ** 2)

In [ ]:
print("\n第二步 打乱、接 mask token、还原（8 个 token 演示）")
restored = shuffle_and_restore(list("ABCDEFGH"))
print(["□" if t is None else t for t in restored])

In [ ]:
print("\n第三步 MAE loss：被遮的一块 [2, 4, 6, 8]，decoder 猜 [-1, -0.5, 0.5, 1]")
target, loss = mae_loss([2, 4, 6, 8], [-1, -0.5, 0.5, 1])
print("归一化目标：", target, " MSE =", round(loss, 3))

In [ ]:
print("\n第四步 DINO：老师 [2, 1, 0]，c = [1, 0, 0]，τt = 0.5；学生 [1, 2, 0]，τs = 1")
p_t, p_s, h = dino_loss([2, 1, 0], [1, 2, 0], np.array([1, 0, 0]), 0.5, 1.0)
print("Pt =", p_t, " Ps =", p_s, " H =", round(h, 3))
print("锐化对比：[2, 1, 0] 温度 1 →", softmax(np.array([2, 1, 0.0])), "；温度 0.5 →", softmax(np.array([2, 1, 0.0]) / 0.5))

In [ ]:
print("\n第五步 EMA：老师 1.0，学生 2.0，λ = 0.996")
print(round(ema(1.0, 2.0, 0.996), 4))

## 3. 代码里才看得到的

- `x.std()` 默认除以 n（总体标准差），PyTorch 的 `std()` 默认除以 n − 1，归一化后的数会差一点；MAE 官方实现用的是 `var` 加一个很小的数再开方。
- DINO 里老师那一边要 `detach()`（不求梯度），NumPy 里没有梯度，这件事在代码里看不出来，只能记住：只有学生被训练，老师靠 EMA 更新。

## 4. 自己改一改

- `random_mask` 的比例改成 0.5、0.9，看剩几块、格子数怎么变。
- 把老师的温度从 0.5 改成 1 和 0.1，看 $P_t$ 变平还是变尖。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 拼图只给四分一，图片啰嗦遮得多。
> 专家只看露出的，助手占位来补全；训完扔掉助手只留专家。
> 只算遮住的那几块，每块归一再比方差。
> 两张裁剪同一图，学生去贴老师的分布。
> 老师是学生的慢平均；减中心、低温度，两招牵制不坍缩。